# Batch Processing

Real-world motion capture datasets contain hundreds or thousands of BVH files. `pybvh.batch` provides utilities for loading directories, validating compatibility, and converting to NumPy.

Before you can batch-process a dataset, two classical problems need to be addressed:

1. **Heterogeneous sources.** Different files may use different up-axis conventions (`+y` vs `+z`), different frame rates, and different skeleton topologies. These have to be **harmonized** before batching.
2. **Shape uniformity.** ML models consume tensors of fixed shape. BVH clips have variable length; we either pad to a common length or keep them as a list.

This tutorial walks through both in order, then puts them together into a complete dataset-preparation pipeline. A third classical step — **per-channel feature normalization** — is an ML-pipeline concern and lives in [pybvh-ml](https://github.com/VictorS-67/pybvh-ml); see the note near the end of this tutorial.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

import warnings

import pybvh
from pybvh import batch
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
bvh_folder = REPO_ROOT / "bvh_data"
output_folder = Path('./output')
output_folder.mkdir(exist_ok=True)

# Loading a directory

`read_bvh_directory()` loads every matching BVH file in a directory and returns a `list[Bvh]`. Files are sorted alphabetically by default (`sort=True`), which matters for reproducibility — shuffling should happen downstream, after deterministic loading.

In [2]:
bvh_list = batch.read_bvh_directory(bvh_folder)

print(f'Loaded {len(bvh_list)} BVH files:')
for i, b in enumerate(bvh_list):
    print(f'  [{i}] {b.joint_count:>2d} joints, {b.frame_count:>3d} frames, '
          f'fps={b.fps:>5.0f}, world_up={b.world_up}')

Loaded 6 BVH files:
  [0] 24 joints,  75 frames, fps=   30, world_up=+z
  [1] 24 joints,  75 frames, fps=   30, world_up=+z
  [2] 23 joints,  61 frames, fps=  120, world_up=+y
  [3] 60 joints, 100 frames, fps=  120, world_up=+z
  [4] 31 joints, 524 frames, fps=  120, world_up=+y
  [5] 24 joints,   1 frames, fps=  120, world_up=+z


/home/victor/projects/pybvh/pybvh/bvh.py:168: UserWarning: Rest pose suggests world up is '+y' but the first animation frame's head-hips direction is closer to '+z'. Using '+z' from the animation data. If this is wrong for your file, set it explicitly via `bvh.world_up = '<axis>'`.
  self._world_up_cached = _infer_world_up(self, warn=warn_on_disagreement)


Notice that the loaded files are heterogeneous: different joint counts, frame rates, and up-axis conventions. We'll unify them below.

## Filtering with patterns

The `pattern` parameter accepts glob patterns. Common uses: restrict to a sub-category (`'*walk*.bvh'`), exclude metadata files (`'subject*.bvh'`), or handle nested structure (`'*/*.bvh'`).

In [3]:
# Only files starting with 'bvh_test' (excludes bvh_example.bvh and standard_skeleton.bvh)
test_files = batch.read_bvh_directory(bvh_folder, pattern='bvh_test*.bvh')
print(f'Loaded {len(test_files)} of {len(bvh_list)} files matching "bvh_test*.bvh"')

Loaded 3 of 6 files matching "bvh_test*.bvh"


## Parallel loading

For large datasets, `parallel=True` uses a thread pool to overlap file I/O. Useful when loading thousands of files; negligible benefit for small fixtures. `max_workers` caps the pool size (defaults to Python's `ThreadPoolExecutor` default, typically CPU count).

In [4]:
bvh_list_parallel = batch.read_bvh_directory(bvh_folder, parallel=True, max_workers=4)
print(f'Loaded {len(bvh_list_parallel)} files in parallel')

Loaded 6 files in parallel


## Robustness: skipping corrupt files

Real datasets sometimes contain malformed or corrupt BVH files — a half-finished export, a stray non-BVH file with a `.bvh` extension, etc. By default `read_bvh_directory` propagates the first failure and aborts the load. Pass `skip_errors=True` to instead emit a `UserWarning` per bad file and return only the successes:

```python
clips = batch.read_bvh_directory('big_dataset/', parallel=True, skip_errors=True)
```

Silent skipping is opt-in precisely because it can hide real problems (you silently lose data). Only enable it when occasional corrupt files are expected and logging the skip is enough.

## Dataset-wide conventions at load time

Two parameters apply a convention uniformly across the whole dataset at load time:

- **`world_up='+z'`** — force every file's `world_up` metadata to the given value, skipping auto-detection. Use when you **know** all your files follow one convention but pybvh's heuristic mis-identifies some (the ~5% edge case). This only sets the metadata; it does **not** rotate the data — for genuine rotation, see *Up-axis unification* below.
- **`lr_mapping={...}`** — apply one explicit left/right joint pair mapping to every file. Useful for datasets with non-standard naming conventions the auto-detect heuristic can't parse.

In [5]:
# Force every loaded file to be interpreted as +z up
bvh_list_zup = batch.read_bvh_directory(bvh_folder, world_up='+z')
print('After world_up="+z" at load:')
for i, b in enumerate(bvh_list_zup):
    print(f'  [{i}] world_up={b.world_up}')

After world_up="+z" at load:
  [0] world_up=+z
  [1] world_up=+z
  [2] world_up=+z
  [3] world_up=+z
  [4] world_up=+z
  [5] world_up=+z


# Harmonizing heterogeneous datasets

When clips come from different sources, three mismatches commonly block batching: **skeleton topology**, **frame rate**, and **up-axis convention**. Each has a corresponding pybvh tool. Apply them in a loop over your clips before `batch_to_numpy`.

## Skeleton unification

Two sub-problems:

- **Different bone proportions, same topology**: `bvh.retarget(reference)` copies bone offsets from a reference skeleton while preserving joint angles. Use when clips share the same joint names and hierarchy but represent differently-sized performers.
- **Different joint counts or names**: `bvh.extract_joints([common_names])` keeps a shared subset, collapsing removed joints' offsets into the nearest kept descendant. See [Tutorial 2](2.Spatial_coordinates.ipynb) for details.

If neither tool applies — e.g., finger-rich files vs. body-only files with no sensible common subset — the practical answer is to **drop the incompatible files**.

In [6]:
# Retarget every compatible clip to a reference skeleton's bone proportions
reference = pybvh.read_bvh_file(bvh_folder / 'standard_skeleton.bvh')
clip_a = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')
clip_b = pybvh.read_bvh_file(bvh_folder / 'bvh_example.bvh')

print('Before retargeting — Spine offset varies:')
print(f'  reference: {reference.nodes[1].offset}')
print(f'  clip_a:    {clip_a.nodes[1].offset}')
print(f'  clip_b:    {clip_b.nodes[1].offset}')

retargeted = [c.retarget(reference) for c in [clip_a, clip_b]]

print('\nAfter retargeting — all match the reference:')
for i, c in enumerate(retargeted):
    print(f'  clip_{chr(ord("a")+i)}:    {c.nodes[1].offset}')

Before retargeting — Spine offset varies:
  reference: [ 0.   -0.38  4.  ]
  clip_a:    [0.     0.     4.4528]
  clip_b:    [0.     0.     4.4528]

After retargeting — all match the reference:
  clip_a:    [ 0.   -0.38  4.  ]
  clip_b:    [ 0.   -0.38  4.  ]


## Frame-rate unification

Clips recorded at different fps can't be batched directly — their feature arrays have different time resolutions. `bvh.resample(target_fps)` uses quaternion SLERP to produce a clip at the target frame rate while preserving the underlying motion.

(This is also the primitive `transforms.perturb_speed` builds on, covered in [Tutorial 5](5.Transforms.ipynb).)

In [7]:
bvh_30 = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')   # 30 fps
bvh_120 = pybvh.read_bvh_file(bvh_folder / 'bvh_test2.bvh')  # 120 fps

print('Before unification:')
for name, b in [('bvh_30', bvh_30), ('bvh_120', bvh_120)]:
    print(f'  {name:7s}  {b.frame_count:>3d} frames @ {b.fps:>4.0f} fps')

target_fps = 30
unified = [b if abs(b.fps - target_fps) < 0.1 else b.resample(target_fps)
           for b in [bvh_30, bvh_120]]

print(f'\nAfter resampling to {target_fps} fps:')
for i, c in enumerate(unified):
    print(f'  clip_{i}:   {c.frame_count:>3d} frames @ {c.fps:>4.0f} fps')

Before unification:
  bvh_30    75 frames @   30 fps
  bvh_120   61 frames @  120 fps

After resampling to 30 fps:
  clip_0:    75 frames @   30 fps
  clip_1:    16 frames @   30 fps


## Up-axis unification

When files come from tools with different up-axis conventions (Maya commonly exports `+y`; Blender commonly `+z`), their world coordinates live in different frames. Visualizing them side-by-side or computing trajectories produces nonsense until they're rotated into a common axis.

`bvh.reorient_world_up(new_up)` rotates the entire scene so the world vertical axis changes, without altering how the character looks (covered in more depth in [Tutorial 5](5.Transforms.ipynb)).

In [8]:
bvh_yup = pybvh.read_bvh_file(bvh_folder / 'bvh_test2.bvh')  # +y
bvh_zup = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')  # +z

print('Before unification:')
for name, b in [('bvh_yup', bvh_yup), ('bvh_zup', bvh_zup)]:
    print(f'  {name:7s}  world_up={b.world_up}')

target_up = '+z'
unified = [b if b.world_up == target_up else b.reorient_world_up(target_up)
           for b in [bvh_yup, bvh_zup]]

print(f'\nAfter reorienting to {target_up}:')
for i, c in enumerate(unified):
    print(f'  clip_{i}:   world_up={c.world_up}')

Before unification:
  bvh_yup  world_up=+y
  bvh_zup  world_up=+z

After reorienting to +z:
  clip_0:   world_up=+z
  clip_1:   world_up=+z


## Harmonizing everything at once

The three subsections above applied `retarget`, `resample`, and `reorient_world_up` one at a time. In practice you'll apply them together to every clip in a dataset, after first checking topology compatibility. `batch.harmonize()` composes all of that behind one call:

- Topology check vs a reference skeleton (drop or raise on mismatch)
- Retarget bone proportions to match the reference
- Resample to a target fps (with a small tolerance to avoid no-op copies)
- Reorient into a target up axis
- Re-express joint angles in a uniform Euler order

Any of `reference`, `target_fps`, `target_world_up`, `target_rest_up`, `target_rest_forward`, `target_euler_order` may be `None` to skip that stage. When clips are dropped, `harmonize` emits **one summary `UserWarning` per call** (not one per dropped clip), or raises `ValueError` immediately with `on_incompatible='raise'`. For workflows `batch.harmonize` doesn't fit — e.g. using `extract_joints` to reduce clips to a common joint subset instead of dropping mismatched files — fall back on the three primitives directly.

In [9]:
reference = pybvh.read_bvh_file(bvh_folder / 'bvh_example.bvh')
raw = [pybvh.read_bvh_file(bvh_folder / name) for name in
       ['bvh_example.bvh', 'bvh_test1.bvh', 'bvh_test2.bvh']]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # quiet the summary drop warning for a tidy cell output
    harmonized = batch.harmonize(
        raw,
        reference=reference,
        target_fps=30,
        target_world_up='+z',
        target_euler_order='XYZ',
        verbose=False,
    )

print(f'In: {len(raw)}  Out: {len(harmonized)} '
      f'(bvh_test2 dropped — different topology)')
for i, c in enumerate(harmonized):
    print(f'  clip {i}: {c.joint_count} joints, {c.frame_count} frames '
          f'@ {c.fps:.0f} fps, up={c.world_up}, '
          f"order={c.euler_orders[0]}")

In: 3  Out: 2 (bvh_test2 dropped — different topology)
  clip 0: 24 joints, 75 frames @ 30 fps, up=+z, order=XYZ
  clip 1: 24 joints, 75 frames @ 30 fps, up=+z, order=XYZ


### Auditing what `harmonize` did

Pass `return_report=True` to also receive a `HarmonizeReport` — a JSON-serializable record of every transformation applied to every kept clip, plus the index, `source_path`, and reason for every dropped one. Useful for embedding alongside a preprocessed dataset so the harmonization trail stays auditable.

In [10]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    harmonized, report = batch.harmonize(
        raw,
        reference=reference,
        target_fps=30,
        target_world_up='+z',
        target_euler_order='XYZ',
        return_report=True,
        verbose=False,
    )

print(f'Kept {len(report.kept_indices)} / dropped {len(report.dropped_indices)}')
for idx, src, stages in zip(report.kept_indices,
                             report.kept_sources,
                             report.applied_stages):
    src_name = Path(src).name if src else f'<index {idx}>'
    print(f'  {src_name}: {stages}')
for idx, src, reason in zip(report.dropped_indices,
                             report.dropped_sources,
                             report.drop_reasons):
    src_name = Path(src).name if src else f'<index {idx}>'
    print(f'  DROPPED {src_name}: {reason}')

Kept 2 / dropped 1
  bvh_example.bvh: {'retarget': 'applied', 'euler_order': '→XYZ'}
  bvh_test1.bvh: {'retarget': 'applied', 'euler_order': '→XYZ'}
  DROPPED bvh_test2.bvh: topology mismatch with reference


# Batch conversion to NumPy

Once clips are harmonized, `batch.batch_to_numpy()` converts a list of `Bvh` objects into NumPy arrays with a flat per-frame feature vector. It validates skeleton compatibility first — mismatched clips raise `ValueError` before any conversion happens.

Build a small demo batch from one clip sliced at different ranges so all share the same skeleton (frame slicing — `bvh[start:stop]` — was introduced in Tutorial 1):

In [11]:
base = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')
clips = [base, base[0:40], base[20:75]]

print(f'Clip frame counts: {[c.frame_count for c in clips]}')

Clip frame counts: [75, 40, 55]


## Variable-length vs. padded output

`pad=False` (default) returns one 2D array per clip — good when clip length is a property of the data (e.g., variable-length sequence models). `pad=True` zero-pads to the longest clip and returns a single 3D tensor — good for fixed-length batching.

In [12]:
arrays = batch.batch_to_numpy(clips, representation='6d')
print(f'pad=False → {type(arrays).__name__} of:')
for i, a in enumerate(arrays):
    print(f'  clip {i}: shape {a.shape}')

padded = batch.batch_to_numpy(clips, representation='6d', pad=True)
print(f'\npad=True  → single array: shape {padded.shape}  (B, F_max, D)')

pad=False → list of:
  clip 0: shape (75, 147)
  clip 1: shape (40, 147)
  clip 2: shape (55, 147)

pad=True  → single array: shape (3, 75, 147)  (B, F_max, D)


## Feature-column layout

The flat feature dimension `D` is structured:

`D = 3 (root position X, Y, Z) + J × rep_dim (flattened joint rotations)`

with `rep_dim` depending on the representation:

| Representation | `rep_dim` per joint | Notes |
|---|---|---|
| `euler`     | 3 | Raw Euler angles, radians |
| `6d`        | 6 | First two columns of rotation matrix (Zhou et al., 2019) |
| `quat`      | 4 | Scalar-first (w, x, y, z) |
| `axisangle` | 3 | Rotation vector; norm = angle in radians |
| `rotmat`    | 9 | Full 3×3 rotation matrix, flattened |

Pass `include_root_pos=False` to drop the 3 leading position columns when your model conditions on rotation only.

In [13]:
print(f'Joint count: {base.joint_count}; expected D = 3 + J × rep_dim\n')
for rep in ['euler', '6d', 'quat', 'axisangle', 'rotmat']:
    arr = batch.batch_to_numpy(clips, representation=rep)[0]
    rep_dim = (arr.shape[1] - 3) // base.joint_count
    print(f'  {rep:11s}  rep_dim = {rep_dim}  →  D = {arr.shape[1]}')

Joint count: 24; expected D = 3 + J × rep_dim

  euler        rep_dim = 3  →  D = 75
  6d           rep_dim = 6  →  D = 147
  quat         rep_dim = 4  →  D = 99
  axisangle    rep_dim = 3  →  D = 75
  rotmat       rep_dim = 9  →  D = 219


## Validation

Three predicates let you check compatibility at the granularity the downstream operation requires:

- **`bvh.matches_hierarchy(other)`** — node names, parent structure, and rest offsets match. Use this when batching to rotation-invariant representations (`'6d'`, `'quat'`, `'rotmat'`), where channel layout is irrelevant. Pass `match_offsets=False` to also accept clips with differing bone proportions (about to be retargeted).
- **`bvh.matches_channels(other)`** — per-joint Euler rotation orders match. Combine with `matches_hierarchy` when batching to `'euler'` or `'axisangle'`, where the channel layout depends on the source Euler order.
- **`bvh.matches_topology(other)`** — conjunction of both. Strictest check; use when every aspect must align.

`batch_to_numpy(...)` picks the right predicate automatically based on the requested representation, and raises `ValueError` with an actionable message (joint name, both orders, source paths where available, recovery hint) on mismatch.

In [14]:
# Soft predicates — no exception, just booleans
ex = pybvh.read_bvh_file(bvh_folder / 'bvh_example.bvh')   # 24 joints
t1 = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')     # 24 joints, same skeleton
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # bvh_test3 emits a rest/animation warning on load
    t3 = pybvh.read_bvh_file(bvh_folder / 'bvh_test3.bvh')  # 60 joints

print(f'bvh_example vs bvh_test1: hierarchy={ex.matches_hierarchy(t1)}, '
      f'channels={ex.matches_channels(t1)}, topology={ex.matches_topology(t1)}')
print(f'bvh_example vs bvh_test3: hierarchy={ex.matches_hierarchy(t3)}, '
      f'channels={ex.matches_channels(t3)}, topology={ex.matches_topology(t3)}')

bvh_example vs bvh_test1: hierarchy=True, channels=True, topology=True
bvh_example vs bvh_test3: hierarchy=False, channels=False, topology=False


In [15]:
# Files with different joint counts cannot batch together
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # bvh_test3 emits a rest/animation warning on load
    incompat_a = pybvh.read_bvh_file(bvh_folder / 'bvh_test1.bvh')  # 24 joints
    incompat_b = pybvh.read_bvh_file(bvh_folder / 'bvh_test3.bvh')  # 60 joints

try:
    batch.batch_to_numpy([incompat_a, incompat_b])
except ValueError as e:
    print(f'Caught: {e}')

Caught: Skeleton hierarchy mismatch between index 0 ('/home/victor/projects/pybvh/bvh_data/bvh_test1.bvh') and index 1 ('/home/victor/projects/pybvh/bvh_data/bvh_test3.bvh'): node count 29 vs 73.


# Normalization

Per-channel z-score normalization — dataset mean/std statistics, apply, reverse — is an ML-pipeline concern, and as of pybvh v0.8.0 it lives in [pybvh-ml](https://github.com/VictorS-67/pybvh-ml):

```python
from pybvh_ml import compute_normalization_stats, normalize_array, denormalize_array
```

The functions work exactly as they used to in `pybvh.batch`: compute stats over unpadded training-split arrays, `(data - mean) / std` to normalize, the reverse to map model output back into BVH units.

# End-to-end pipeline

Complete dataset-preparation workflow, combining everything above:

In [16]:
# 1. Load raw files
raw = batch.read_bvh_directory(bvh_folder, pattern='bvh_*.bvh')
print(f'Step 1 — Loaded {len(raw)} files')

# 2. Harmonize (topology check / retarget / resample / reorient / Euler order)
#    in one call. Clips incompatible with the reference skeleton are dropped;
#    `harmonize` emits one summary UserWarning at end of call when that happens.
#    We use bvh_example as the canonical rig here.
reference = pybvh.read_bvh_file(bvh_folder / 'bvh_example.bvh')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # quiet the summary drop warning for tutorial output
    harmonized = batch.harmonize(
        raw,
        reference=reference,
        target_fps=30,
        target_world_up='+z',
        target_euler_order='XYZ',
        verbose=False,
    )
print(f'Step 2 — Harmonized, kept {len(harmonized)} of {len(raw)} clips')

# 3. Convert to arrays
arrays = batch.batch_to_numpy(harmonized, representation='6d')
print(f'Step 3 — Converted, D = {arrays[0].shape[1]}')

# 4. Save
np.savez(output_folder / 'dataset.npz',
         **{f'clip_{i}': a for i, a in enumerate(arrays)})
print(f'Step 4 — Saved {len(arrays)} clips to {output_folder}/')

# Clean up (for tutorial reruns)
(output_folder / 'dataset.npz').unlink()

Step 1 — Loaded 4 files
Step 2 — Harmonized, kept 2 of 4 clips
Step 3 — Converted, D = 147
Step 4 — Saved 2 clips to output/


/home/victor/projects/pybvh/pybvh/bvh.py:168: UserWarning: Rest pose suggests world up is '+y' but the first animation frame's head-hips direction is closer to '+z'. Using '+z' from the animation data. If this is wrong for your file, set it explicitly via `bvh.world_up = '<axis>'`.
  self._world_up_cached = _infer_world_up(self, warn=warn_on_disagreement)


For ML-framework-specific downstream work — per-channel normalization, PyTorch `Dataset` classes, DataLoaders, collate functions, augmentation pipelines, HDF5 packing — see [pybvh-ml](https://github.com/VictorS-67/pybvh-ml), the companion library that builds on top of pybvh.

# Summary

| Function / method | Purpose |
|---|---|
| `batch.read_bvh_directory(dir)` | Load all matching BVH files |
| `bvh.retarget(reference)` | Copy bone offsets from a reference skeleton |
| `bvh.resample(target_fps)` | Resample to a target frame rate (SLERP) |
| `bvh.reorient_world_up(axis)` | Rotate scene into a common up axis |
| `batch.batch_to_numpy(clips)` | Convert to `(F_i, D)` arrays or `(B, F_max, D)` tensor |

Key parameters (selection — see the [feature-export guide](https://victors-67.github.io/pybvh/guide/feature-export/) for the full reference):

- **`read_bvh_directory`**: `pattern`, `sort`, `parallel`, `max_workers`, `world_up`, `lr_mapping`
- **`batch_to_numpy`**: `representation`, `include_root_pos`, `pad`, `pad_value`

# What's next

- [Tutorial 5 — Transforms](5.Transforms.ipynb) covered augmentation transforms (mirror, yaw rotation, noise, speed). Apply them to the harmonized clips as a stochastic augmentation step before array conversion.
- [Tutorial 6 — Motion Features](6.Features.ipynb) covered velocities, foot contacts, and `to_feature_array()` — richer alternatives to raw joint angles.
- [Tutorial 8 — Motion Descriptors](8.Motion_descriptors.ipynb) is a standalone deep-dive into higher-level descriptors (trajectory geometry, smoothness, gait, SE(3) features) — useful as engineered features alongside the arrays exported here.
- For ML-framework integration (PyTorch, TensorFlow), see [pybvh-ml](https://github.com/VictorS-67/pybvh-ml).